In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Dataset location in My Drive
DATASET_PATH = Path('/content/drive/MyDrive/Ultrasound_Voice_Dataset')

# Keep this class order consistent during training and inference
CLASS_NAMES = [
    '00_capture_image',
    '01_capture_cine',
    '02_increase_depth',
    '03_decrease_depth',
    '04_increase_gain',
    '05_decrease_gain',
    '06_freeze',
    '07_toggle_auto_gain',
    '08_center_guide',
    '09_color_doppler',
    '10_power_doppler',
    '11_needle_enhance',
    '12_b_mode',
    '13_m_mode',
    '14_negative',
]

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.m4a', '.ogg'}

if not DATASET_PATH.is_dir():
    raise FileNotFoundError(f'Dataset folder not found: {DATASET_PATH}')

total_audio = 0

print(f'Dataset: {DATASET_PATH}\n')
for class_name in CLASS_NAMES:
    folder = DATASET_PATH / class_name

    if not folder.is_dir():
        print(f'Missing folder: {class_name}')
        continue

    audio_files = sorted(
        file for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS
    )
    total_audio += len(audio_files)
    print(f'{class_name:24s}  {len(audio_files):3d} audio files')

print(f'\nTotal: {total_audio} audio files')

Mounted at /content/drive
Dataset: /content/drive/MyDrive/Ultrasound_Voice_Dataset

00_capture_image          100 audio files
01_capture_cine           100 audio files
02_increase_depth         100 audio files
03_decrease_depth         100 audio files
04_increase_gain          100 audio files
05_decrease_gain          100 audio files
06_freeze                 100 audio files
07_toggle_auto_gain       100 audio files
08_center_guide           100 audio files
09_color_doppler          100 audio files
10_power_doppler          100 audio files
11_needle_enhance         100 audio files
12_b_mode                 100 audio files
13_m_mode                 100 audio files
14_negative               129 audio files

Total: 1529 audio files


In [ ]:
import librosa
import numpy as np
from collections import Counter

audio_records = []
problem_files = []

for class_name in CLASS_NAMES:
    folder = DATASET_PATH / class_name
    files = sorted(
        file for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS
    )

    for file in files:
        try:
            waveform, sample_rate = librosa.load(
                file,
                sr=None,
                mono=True
            )

            if len(waveform) == 0 or not np.isfinite(waveform).all():
                raise ValueError("Empty audio or invalid sample values")

            duration = len(waveform) / sample_rate

            audio_records.append({
                "path": str(file),
                "class_name": class_name,
                "filename": file.name,
                "duration": duration,
                "sample_rate": sample_rate,
            })

        except Exception as error:
            problem_files.append((str(file), str(error)))

durations = np.array([record["duration"] for record in audio_records])
sample_rates = Counter(record["sample_rate"] for record in audio_records)

print(f"Readable files: {len(audio_records)}")
print(f"Problem files: {len(problem_files)}")
print(f"Sample rates: {dict(sample_rates)}")

if len(durations) > 0:
    print(f"Shortest duration: {durations.min():.2f} s")
    print(f"Longest duration:  {durations.max():.2f} s")
    print(f"Files over 2.0 s:  {np.sum(durations > 2.0)}")

print("\nExample filenames:")
for class_name in [CLASS_NAMES[0], CLASS_NAMES[6], CLASS_NAMES[-1]]:
    examples = [
        record["filename"]
        for record in audio_records
        if record["class_name"] == class_name
    ][:5]
    print(f"{class_name}: {examples}")

if problem_files:
    print("\nFiles that could not be read:")
    for path, error in problem_files[:20]:
        print(f"{path} -> {error}")

Readable files: 1529
Problem files: 0
Sample rates: {44100: 34, 24000: 1495}
Shortest duration: 0.70 s
Longest duration:  2.74 s
Files over 2.0 s:  369

Example filenames:
00_capture_image: ['Capture Image_1.mp3', 'Capture Image_10.mp3', 'Capture Image_100.mp3', 'Capture Image_11.mp3', 'Capture Image_12.mp3']
06_freeze: ['Freeze_1.mp3', 'Freeze_10.mp3', 'Freeze_100.mp3', 'Freeze_11.mp3', 'Freeze_12.mp3']
14_negative: ['Negative_1.mp3', 'Negative_10.mp3', 'Negative_100.mp3', 'Negative_101.mp3', 'Negative_102.mp3']


In [ ]:
import subprocess

bad_file = DATASET_PATH / "01_capture_cine" / "Capture Cine_8.mp3"

print("Problem file")
print(f"Path: {bad_file}")
print(f"Exists: {bad_file.exists()}")
print(f"Size: {bad_file.stat().st_size if bad_file.exists() else 'N/A'} bytes")

if bad_file.exists():
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_format", "-show_streams", str(bad_file)],
        capture_output=True,
        text=True
    )
    print(f"ffprobe exit code: {result.returncode}")
    print((result.stdout or result.stderr)[:1500])

print("\nLongest readable files")
for record in sorted(audio_records, key=lambda item: item["duration"], reverse=True)[:10]:
    print(f'{record["duration"]:.2f} s | {record["class_name"]} | {record["filename"]}')

Problem file
Path: /content/drive/MyDrive/Ultrasound_Voice_Dataset/01_capture_cine/Capture Cine_8.mp3
Exists: True
Size: 11232 bytes
ffprobe exit code: 0
[STREAM]
index=0
codec_name=mp3
codec_long_name=MP3 (MPEG audio layer 3)
profile=unknown
codec_type=audio
codec_tag_string=[0][0][0][0]
codec_tag=0x0000
sample_fmt=fltp
sample_rate=24000
channels=1
channel_layout=mono
bits_per_sample=0
initial_padding=0
id=N/A
r_frame_rate=0/0
avg_frame_rate=0/0
time_base=1/14112000
start_pts=0
start_time=0.000000
duration_ts=26417664
duration=1.872000
bit_rate=48000
max_bit_rate=N/A
bits_per_raw_sample=N/A
nb_frames=N/A
nb_read_frames=N/A
nb_read_packets=N/A
DISPOSITION:default=0
DISPOSITION:dub=0
DISPOSITION:original=0
DISPOSITION:comment=0
DISPOSITION:lyrics=0
DISPOSITION:karaoke=0
DISPOSITION:forced=0
DISPOSITION:hearing_impaired=0
DISPOSITION:visual_impaired=0
DISPOSITION:clean_effects=0
DISPOSITION:attached_pic=0
DISPOSITION:timed_thumbnails=0
DISPOSITION:non_diegetic=0
DISPOSITION:captions=0
DI

In [ ]:
import random
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

SEED = 42
SAMPLE_RATE = 16000
DURATION = 3.0
TARGET_LENGTH = int(SAMPLE_RATE * DURATION)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

train_records, validation_records = train_test_split(
    audio_records,
    test_size=0.20,
    random_state=SEED,
    stratify=[record["class_name"] for record in audio_records]
)

class UltrasoundDataset(Dataset):
    def __init__(self, records):
        self.records = records
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=512,
            win_length=400,
            hop_length=160,
            n_mels=40
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        waveform_np, _ = librosa.load(
            record["path"],
            sr=SAMPLE_RATE,
            mono=True
        )
        waveform = torch.from_numpy(waveform_np).float()

        if waveform.numel() < TARGET_LENGTH:
            missing = TARGET_LENGTH - waveform.numel()
            left = missing // 2
            waveform = F.pad(waveform, (left, missing - left))
        else:
            waveform = waveform[:TARGET_LENGTH]

        # Adaptive PCEN receives nonnegative Mel energies, not Log-Mel values.
        mel = self.mel_transform(waveform)
        label = CLASS_NAMES.index(record["class_name"])

        return mel, label

train_dataset = UltrasoundDataset(train_records)
validation_dataset = UltrasoundDataset(validation_records)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

sample_mel, sample_labels = next(iter(train_loader))

print(f"Training files: {len(train_dataset)}")
print(f"Validation files: {len(validation_dataset)}")
print(f"Mel batch shape: {tuple(sample_mel.shape)}")
print(f"Minimum Mel value: {sample_mel.min().item():.6f}")

Training files: 1223
Validation files: 306
Mel batch shape: (8, 40, 301)
Minimum Mel value: 0.000000


In [ ]:
class AdaptivePCEN(nn.Module):
    def __init__(self, num_bands=40, hidden_dim=32, eps=1e-5):
        super().__init__()
        self.eps = eps

        self.controller_gru = nn.GRU(
            input_size=num_bands * 2,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.controller_fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_bands * 2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = torch.clamp(x, min=1e-5)
        batch_size, num_bands, time_steps = x.shape

        # Smooth the Mel energy over time.
        smoothing = 0.025
        smoothed_frames = []
        current = x[:, :, 0]

        for time_index in range(time_steps):
            if time_index > 0:
                current = (
                    (1 - smoothing) * current
                    + smoothing * x[:, :, time_index]
                )
            smoothed_frames.append(current)

        smoothed_energy = torch.stack(smoothed_frames, dim=2)

        # Predict time-varying PCEN parameters from consecutive frames.
        previous_frames = F.pad(x[:, :, :-1], (1, 0))
        controller_input = torch.cat(
            [x, previous_frames],
            dim=1
        ).transpose(1, 2)

        gru_output, _ = self.controller_gru(controller_input)
        parameters = self.controller_fc(gru_output)

        alpha = parameters[:, :, :num_bands].transpose(1, 2)
        gamma_raw = parameters[:, :, num_bands:].transpose(1, 2)

        alpha = torch.clamp(alpha, min=0.01, max=0.99)
        gamma = 0.2 + 0.7 * gamma_raw

        denominator = torch.pow(self.eps + smoothed_energy, alpha)
        normalized = x / torch.clamp(denominator, min=1e-5)

        return torch.pow(
            torch.clamp(normalized, min=1e-5),
            gamma
        )


class UltrasoundAdaptivePCENModel(nn.Module):
    def __init__(self, num_classes=15):
        super().__init__()

        self.apcen = AdaptivePCEN(num_bands=40)

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, mel):
        x = self.apcen(mel).unsqueeze(1)

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x).flatten(start_dim=1)
        x = self.dropout(x)

        return self.classifier(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UltrasoundAdaptivePCENModel(
    num_classes=len(CLASS_NAMES)
).to(device)

with torch.no_grad():
    sample_output = model(sample_mel.to(device))

print(f"Device: {device}")
print(f"Model output shape: {tuple(sample_output.shape)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: cpu
Model output shape: (8, 15)
Model parameters: 51,103


In [ ]:
import copy

EPOCHS = 15
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_accuracy = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for mel, labels in train_loader:
        mel = mel.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(mel)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    validation_correct = 0
    validation_total = 0

    with torch.no_grad():
        for mel, labels in validation_loader:
            mel = mel.to(device)
            labels = labels.to(device)

            logits = model(mel)
            validation_correct += (
                logits.argmax(1) == labels
            ).sum().item()
            validation_total += labels.size(0)

    train_accuracy = train_correct / train_total
    validation_accuracy = validation_correct / validation_total

    if validation_accuracy > best_accuracy:
        best_accuracy = validation_accuracy
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss_sum / train_total:.4f} | "
        f"Train: {train_accuracy:.2%} | "
        f"Validation: {validation_accuracy:.2%}"
    )

model.load_state_dict(best_state)

MODEL_PATH = DATASET_PATH.parent / "ultrasound_apcen_kws.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "sample_rate": SAMPLE_RATE,
    "duration": DURATION,
    "n_mels": 40,
    "architecture": "AdaptivePCEN_GRU_CNN",
}, MODEL_PATH)

print(f"\nBest validation accuracy: {best_accuracy:.2%}")
print(f"Saved model: {MODEL_PATH}")

Epoch 01/15 | Loss: 2.6442 | Train: 10.30% | Validation: 22.88%
Epoch 02/15 | Loss: 2.4216 | Train: 22.40% | Validation: 25.82%
Epoch 03/15 | Loss: 2.1247 | Train: 30.09% | Validation: 43.79%
Epoch 04/15 | Loss: 1.8841 | Train: 34.75% | Validation: 42.16%
Epoch 05/15 | Loss: 1.6948 | Train: 43.25% | Validation: 49.67%
Epoch 06/15 | Loss: 1.5172 | Train: 49.80% | Validation: 58.50%
Epoch 07/15 | Loss: 1.3378 | Train: 54.78% | Validation: 57.19%
Epoch 08/15 | Loss: 1.2376 | Train: 60.51% | Validation: 67.97%
Epoch 09/15 | Loss: 1.0863 | Train: 64.51% | Validation: 74.84%
Epoch 10/15 | Loss: 0.9999 | Train: 68.27% | Validation: 72.88%
Epoch 11/15 | Loss: 0.9040 | Train: 70.32% | Validation: 82.03%
Epoch 12/15 | Loss: 0.8361 | Train: 73.34% | Validation: 82.03%
Epoch 13/15 | Loss: 0.8084 | Train: 75.14% | Validation: 84.97%
Epoch 14/15 | Loss: 0.7197 | Train: 78.74% | Validation: 83.66%
Epoch 15/15 | Loss: 0.6085 | Train: 82.50% | Validation: 91.18%

Best validation accuracy: 91.18%
Saved 

In [ ]:
!pip -q install edge-tts

import edge_tts

voices = await edge_tts.list_voices()

english_voices = sorted(
    voice["ShortName"]
    for voice in voices
    if voice["Locale"].lower().startswith("en-")
)

print(f"Available English voices: {len(english_voices)}")
print("\nFirst 20 voices:")
for voice in english_voices[:20]:
    print(voice)

Available English voices: 47

First 20 voices:
en-AU-NatashaNeural
en-AU-WilliamMultilingualNeural
en-CA-ClaraNeural
en-CA-LiamNeural
en-GB-LibbyNeural
en-GB-MaisieNeural
en-GB-RyanNeural
en-GB-SoniaNeural
en-GB-ThomasNeural
en-HK-SamNeural
en-HK-YanNeural
en-IE-ConnorNeural
en-IE-EmilyNeural
en-IN-NeerjaExpressiveNeural
en-IN-NeerjaNeural
en-IN-PrabhatNeural
en-KE-AsiliaNeural
en-KE-ChilembaNeural
en-NG-AbeoNeural
en-NG-EzinneNeural


In [ ]:
import base64
import json
import subprocess
from pathlib import Path

import librosa
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio.transforms as T

from google.colab import output
from IPython.display import Audio, Javascript, display

RECORD_JS = """
async function recordCommand() {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

    const formats = [
        "audio/webm;codecs=opus",
        "audio/webm",
        "audio/ogg;codecs=opus",
        "audio/mp4"
    ];

    const selectedFormat = formats.find(
        format => MediaRecorder.isTypeSupported(format)
    );

    const recorder = selectedFormat
        ? new MediaRecorder(stream, { mimeType: selectedFormat })
        : new MediaRecorder(stream);

    const chunks = [];

    recorder.ondataavailable = event => {
        if (event.data.size > 0) chunks.push(event.data);
    };

    const recordingFinished = new Promise(resolve => {
        recorder.onstop = resolve;
    });

    const status = document.createElement("div");
    status.style.cssText =
        "padding:12px;font-size:20px;font-weight:bold;" +
        "background:#eef2ff;border-radius:8px;";
    document.body.appendChild(status);

    for (let count = 2; count >= 1; count--) {
        status.textContent = `Get ready: ${count}`;
        await new Promise(resolve => setTimeout(resolve, 1000));
    }

    status.textContent = "Recording now — say one command";
    recorder.start();

    await new Promise(resolve => setTimeout(resolve, 5000));

    recorder.stop();
    await recordingFinished;
    stream.getTracks().forEach(track => track.stop());

    status.textContent = "Recording finished";

    const blob = new Blob(chunks, { type: recorder.mimeType });

    const dataUrl = await new Promise(resolve => {
        const reader = new FileReader();
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
    });

    status.remove();

    return JSON.stringify({
        dataUrl: dataUrl,
        mimeType: recorder.mimeType
    });
}
"""

def record_and_analyze():
    required = [
        "model",
        "CLASS_NAMES",
        "SAMPLE_RATE",
        "TARGET_LENGTH",
        "device"
    ]
    missing = [name for name in required if name not in globals()]

    if missing:
        raise RuntimeError(
            "Load the saved model first. Missing: " + ", ".join(missing)
        )

    display(Javascript(RECORD_JS))
    result = json.loads(output.eval_js("recordCommand()"))

    mime_type = result["mimeType"]

    if "mp4" in mime_type:
        extension = ".mp4"
    elif "ogg" in mime_type:
        extension = ".ogg"
    else:
        extension = ".webm"

    source_path = Path("/content/recorded_command" + extension)
    wav_path = Path("/content/recorded_command.wav")

    source_path.write_bytes(
        base64.b64decode(result["dataUrl"].split(",", 1)[1])
    )

    conversion = subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(source_path),
            "-ac", "1",
            "-ar", str(SAMPLE_RATE),
            str(wav_path)
        ],
        capture_output=True,
        text=True
    )

    if conversion.returncode != 0:
        raise RuntimeError(
            "Audio conversion failed: " + conversion.stderr
        )

    full_audio, _ = librosa.load(
        str(wav_path),
        sr=SAMPLE_RATE,
        mono=True
    )

    print(f"Recorded duration: {len(full_audio) / SAMPLE_RATE:.2f} s")
    print("Full recording:")
    display(Audio(full_audio, rate=SAMPLE_RATE))

    if np.max(np.abs(full_audio)) < 0.005:
        print("The recording is very quiet. Check microphone input.")
        return

    speech_intervals = librosa.effects.split(
        full_audio,
        top_db=30,
        frame_length=512,
        hop_length=128
    )

    if len(speech_intervals) == 0:
        print("No speech segment was detected.")
        return

    # Select the longest detected speech segment.
    start, end = max(
        speech_intervals,
        key=lambda interval: interval[1] - interval[0]
    )



    # Keep a short margin around the detected speech.
    margin = int(0.15 * SAMPLE_RATE)
    start = max(0, start - margin)
    end = min(len(full_audio), end + margin)

    speech_audio = full_audio[start:end]

    print(
        f"Selected interval: "
        f"{start / SAMPLE_RATE:.2f}–{end / SAMPLE_RATE:.2f} s"
    )
    print("Audio sent to the model:")
    display(Audio(speech_audio, rate=SAMPLE_RATE))

    if len(speech_audio) > TARGET_LENGTH:
        print(
            "The selected audio exceeds the model's 3-second input. "
            "Record one shorter command."
        )
        return

    waveform = torch.from_numpy(speech_audio).float()

    missing_samples = TARGET_LENGTH - waveform.numel()
    left_padding = missing_samples // 2
    waveform = F.pad(
        waveform,
        (left_padding, missing_samples - left_padding)
    )

    mel_transform = T.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=512,
        win_length=400,
        hop_length=160,
        n_mels=40
    )

    mel = mel_transform(waveform).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(mel), dim=1)[0]
        values, indices = torch.topk(probabilities, k=3)

    print("\nTop predictions:")
    for rank, (value, index) in enumerate(
        zip(values, indices),
        start=1
    ):
        print(
            f"{rank}. {CLASS_NAMES[index.item()]:24s} "
            f"{value.item():.2%}"
        )

    print(f"\nSaved recording: {wav_path}")

record_and_analyze()

<IPython.core.display.Javascript object>

Recorded duration: 5.94 s
Full recording:


Selected interval: 0.00–1.89 s
Audio sent to the model:



Top predictions:
1. 07_toggle_auto_gain      92.13%
2. 10_power_doppler         4.26%
3. 09_color_doppler         3.19%

Saved recording: /content/recorded_command.wav


In [ ]:
import base64
import json
import subprocess
from pathlib import Path

import librosa
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio.transforms as T

from google.colab import output
from IPython.display import Audio, Javascript, display


RECORD_JS = """
async function recordCommand() {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

    const formats = [
        "audio/webm;codecs=opus",
        "audio/webm",
        "audio/ogg;codecs=opus",
        "audio/mp4"
    ];

    const selectedFormat = formats.find(
        format => MediaRecorder.isTypeSupported(format)
    );

    const recorder = selectedFormat
        ? new MediaRecorder(stream, { mimeType: selectedFormat })
        : new MediaRecorder(stream);

    const chunks = [];

    recorder.ondataavailable = event => {
        if (event.data.size > 0) {
            chunks.push(event.data);
        }
    };

    const recordingFinished = new Promise(resolve => {
        recorder.onstop = resolve;
    });

    const status = document.createElement("div");
    status.style.cssText =
        "padding:12px;font-size:20px;font-weight:bold;" +
        "background:#eef2ff;border-radius:8px;";
    document.body.appendChild(status);

    for (let count = 2; count >= 1; count--) {
        status.textContent = `Get ready: ${count}`;
        await new Promise(resolve => setTimeout(resolve, 1000));
    }

    status.textContent = "Recording now — say one command";
    recorder.start();

    await new Promise(resolve => setTimeout(resolve, 5000));

    recorder.stop();
    await recordingFinished;

    stream.getTracks().forEach(track => track.stop());
    status.textContent = "Recording finished";

    const blob = new Blob(chunks, { type: recorder.mimeType });

    const dataUrl = await new Promise(resolve => {
        const reader = new FileReader();
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
    });

    status.remove();

    return JSON.stringify({
        dataUrl: dataUrl,
        mimeType: recorder.mimeType
    });
}
"""


def record_and_analyze():
    required_variables = [
        "model",
        "CLASS_NAMES",
        "SAMPLE_RATE",
        "TARGET_LENGTH",
        "device",
    ]

    missing_variables = [
        name for name in required_variables
        if name not in globals()
    ]

    if missing_variables:
        raise RuntimeError(
            "Load the saved model first. Missing: "
            + ", ".join(missing_variables)
        )

    display(Javascript(RECORD_JS))
    result = json.loads(output.eval_js("recordCommand()"))

    mime_type = result["mimeType"]

    if "mp4" in mime_type:
        extension = ".mp4"
    elif "ogg" in mime_type:
        extension = ".ogg"
    else:
        extension = ".webm"

    source_path = Path("/content/recorded_command" + extension)
    wav_path = Path("/content/recorded_command.wav")

    audio_bytes = base64.b64decode(
        result["dataUrl"].split(",", 1)[1]
    )
    source_path.write_bytes(audio_bytes)

    conversion = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            str(source_path),
            "-ac",
            "1",
            "-ar",
            str(SAMPLE_RATE),
            str(wav_path),
        ],
        capture_output=True,
        text=True,
    )

    if conversion.returncode != 0:
        raise RuntimeError(
            f"Audio conversion failed: {conversion.stderr}"
        )

    full_audio, _ = librosa.load(
        str(wav_path),
        sr=SAMPLE_RATE,
        mono=True,
    )

    print(f"Recorded duration: {len(full_audio) / SAMPLE_RATE:.2f} s")
    print("Full recording:")
    display(Audio(full_audio, rate=SAMPLE_RATE))

    if len(full_audio) == 0 or np.max(np.abs(full_audio)) < 0.005:
        print("The recording is empty or very quiet.")
        return

    speech_intervals = librosa.effects.split(
        full_audio,
        top_db=30,
        frame_length=512,
        hop_length=128,
    )

    if len(speech_intervals) == 0:
        print("No speech segment was detected.")
        return

    # Keep all detected speech, including pauses between words.
    start = speech_intervals[0][0]
    end = speech_intervals[-1][1]

    # Add a short margin around the utterance.
    margin = int(0.15 * SAMPLE_RATE)
    start = max(0, start - margin)
    end = min(len(full_audio), end + margin)

    speech_audio = full_audio[start:end]

    print(
        f"Selected interval: "
        f"{start / SAMPLE_RATE:.2f}–{end / SAMPLE_RATE:.2f} s"
    )
    print("Audio sent to the model:")
    display(Audio(speech_audio, rate=SAMPLE_RATE))

    if len(speech_audio) > TARGET_LENGTH:
        print(
            "The selected audio is longer than the model input. "
            "Record one shorter command."
        )
        return

    waveform = torch.from_numpy(speech_audio).float()

    missing_samples = TARGET_LENGTH - waveform.numel()
    left_padding = missing_samples // 2
    right_padding = missing_samples - left_padding

    waveform = F.pad(
        waveform,
        (left_padding, right_padding),
    )

    mel_transform = T.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=512,
        win_length=400,
        hop_length=160,
        n_mels=40,
    )

    mel = mel_transform(waveform).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        logits = model(mel)
        probabilities = torch.softmax(logits, dim=1)[0]
        values, indices = torch.topk(probabilities, k=3)

    print("\nTop predictions:")

    for rank, (value, index) in enumerate(
        zip(values, indices),
        start=1,
    ):
        print(
            f"{rank}. {CLASS_NAMES[index.item()]:24s} "
            f"{value.item():.2%}"
        )

    print(f"\nSaved recording: {wav_path}")


record_and_analyze()

<IPython.core.display.Javascript object>

Recorded duration: 4.98 s
Full recording:


Selected interval: 0.31–1.59 s
Audio sent to the model:



Top predictions:
1. 01_capture_cine          39.87%
2. 03_decrease_depth        35.35%
3. 05_decrease_gain         10.43%

Saved recording: /content/recorded_command.wav
